# WeekendPulse Reels — Batch Renderer

Turn today's **reel-approved** news posts into short, ~16s vertical Facebook
Reels, then **post them to the Page**. One reel per story. Source of truth is
`manifest.json`: the bot records each post there (post_id, title, description,
reel approval) and the notebook updates each entry after it renders + posts,
recording the voice + emotion used and the returned reel video id.

## Pipeline (per reel)
1. Ken Burns pan/zoom over the article's **real photo** (branded fallback card if none).
2. **Chatterbox-Nano** TTS narration of the AI's `reel_blurb` — a **random
   female voice**, precise emotion (`neutral` / `excited` / `surprised`).
3. Whisper-aligned **captions** burned in.
4. A **title card** (vibrant orange + white) fades in over the body.
5. Music from the repo `music/` folder starts at position 0 and is **cut at
   narration end** (loopable head, auto-ducked under the voice).
6. A **~0.4s crossfade** into the fixed 4s `outro.mp4`.
7. **Post cell** uploads each rendered MP4 to the Page via the Graph API
   `videos` endpoint and records the result in `manifest.json`.

## Voices
- **10 female voices** are auto-downloaded from the public
  `OwenTyme/voice-zero` `voices-emotion/` pool (each is a folder of emotion
  clips, e.g. `emily_cripps/excited.flac`).
- Every reel uses a **random female voice**. The AI's `reel_emotion`
  (`neutral` / `excited` / `surprised`) picks that voice's matching clip; if
  missing it falls back to the voice's `neutral.flac`. A voice is never pinned.
- The voice + emotion used are recorded in `manifest.json` so you can see what
  performs better.

## Usage
- **Run all cells in order.** Only Run/Render and Preview are heavy.
- Rendering needs the **T4 GPU** runtime (Chatterbox-Nano). First run downloads
  ~2.9 GB of model weights + 10 female voice clips + whisper model.
- The **Post** cell needs two tokens **at runtime** (paste them, or set Colab
  Secrets): the **FB page token** (posts reels) and the **GitHub repo token**
  (writes `manifest.json` back). Neither is ever baked into the notebook.
- An article is **not picked again** once its reel is posted (the manifest's
  `reel_posted` flag). Re-render a story is fine until you post it.

> TIP: shorter is better. Keep each reel ~16s (soft target), anything under
> ~25s is fine. Very long blurb results in a long reel — consider editing the
> blurb in `reels_batch.txt` before you re-render.

In [ ]:
# Cell 2 — Environment. Keep cell order; run all.
import subprocess, sys, os

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, check=True, **kw)

print("Installing dependencies ...")
sh("apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg >/dev/null")
pip_base = [sys.executable, "-m", "pip", "-q", "install"]

# core: TTS runtime + torch extras
sh("-c", " ".join([*pip_base,
                  "chatterbox", "pyloudnorm", "torchaudio",
                  "faster-whisper", "torch",
                  "librosa", "soundfile", "pydub", "Pillow",
                  "ipython", "IPython"]))

print("ffmpeg:", sh("ffmpeg -version | head -n1", capture_output=True, text=True).stdout.strip())
print("Setup done.")


In [ ]:
# Cell — install the renderer modules (reel_render.py, align.py, enhance.py).
import base64
from pathlib import Path
SRC = Path('/content/weekendpulse_reels/story_src')
SRC.mkdir(parents=True, exist_ok=True)
SRC.joinpath('align.py').write_text(
    base64.b64decode('...').decode('utf-8'))
SRC.joinpath('enhance.py').write_text(
    base64.b64decode('IiIiQXVkaW8gcG9zdC1wcm9jZXNzaW5nIHBpcGVsaW5lIGZvciBDaGF0dGVyYm94IE5hbm8gb3V0cHV0Lg0KDQpTdGFnZXMgKGVhY2ggaW5kZXBlbmRlbnRseSBza2lwcGFibGUgb24gZmFpbHVyZSk6DQogIDEuIExhdmFTUiAgIOKAlCBzcGVlY2ggZW5oYW5jZW1lbnQgKHdhcm10aCwgYmFuZHdpZHRoLCBjbGFyaXR5KQ0KICAyLiBSTk5vaXNlICDigJQgYXJ0aWZhY3QgcmVtb3ZhbCAobWV0YWxsaWMgaGlzcywgY2xpY2tzKQ0KICAzLiBhdXRvLWVkaXRvciDigJQgdHJpbSBkZWFkIHNpbGVuY2UgYW5kIHN0dXR0ZXJzDQogIDQuIEZGbXBlZyAgIOKAlCBtYXN0ZXJpbmcgKEVRLCBjb21wcmVzc2lvbiwgTFVGUyBub3JtYWxpemF0aW9uKQ0KIiIiDQoNCmltcG9ydCBvcw0KaW1wb3J0IHNodXRpbA0KaW1wb3J0IHN1YnByb2Nlc3MNCmltcG9ydCB0aW1lDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KDQpfTEFWQV9NT0RFTCA9IE5vbmUNCl9MQVZBX0xPQ0sgPSBOb25lDQoNCg0KZGVmIF9nZXRfbGF2YSgpOg0KICAgICIiIkxhenkgc2luZ2xldG9uIGZvciBMYXZhU1IgbW9kZWwgKH41ME1CLCBsb2FkcyBvbmNlKS4iIiINCiAgICBnbG9iYWwgX0xBVkFfTU9ERUwsIF9MQVZBX0xPQ0sNCiAgICBpZiBfTEFWQV9MT0NLIGlzIE5vbmU6DQogICAgICAgIGltcG9ydCB0aHJlYWRpbmcNCiAgICAgICAgX0xBVkFfTE9DSyA9IHRocmVhZGluZy5Mb2NrKCkNCiAgICB3aXRoIF9MQVZBX0xPQ0s6DQogICAgICAgIGlmIF9MQVZBX01PREVMIGlzIE5vbmU6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgZnJvbSBMYXZhU1IubW9kZWwgaW1wb3J0IExhdmFFbmhhbmNlMg0KICAgICAgICAgICAgICAgIF9MQVZBX01PREVMID0gTGF2YUVuaGFuY2UyKCJZYXRoYXJ0aFMvTGF2YVNSIiwgImNwdSIpDQogICAgICAgICAgICAgICAgcHJpbnQoIltlbmhhbmNlXSBMYXZhU1IgbW9kZWwgbG9hZGVkIikNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgICAgICBwcmludChmIltlbmhhbmNlXSBMYXZhU1IgbG9hZCBmYWlsZWQ6IHtlfSIpDQogICAgICAgICAgICAgICAgX0xBVkFfTU9ERUwgPSBGYWxzZSAgIyBzZW50aW5lbDogZG8gbm90IHJldHJ5DQogICAgcmV0dXJuIF9MQVZBX01PREVMIGlmIF9MQVZBX01PREVMIGlzIG5vdCBGYWxzZSBlbHNlIE5vbmUNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBTdGFnZSAxOiBMYXZhU1Igc3BlZWNoIGVuaGFuY2VtZW50DQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KZGVmIF9zdGFnZV9sYXZhKHNyYywgZHN0KToNCiAgICBsYXZhID0gX2dldF9sYXZhKCkNCiAgICBpZiBsYXZhIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBGYWxzZQ0KICAgIGltcG9ydCBzb3VuZGZpbGUgYXMgc2YNCiAgICBhdWRpbywgX3NyID0gbGF2YS5sb2FkX2F1ZGlvKHN0cihzcmMpKQ0KICAgIG91dCA9IGxhdmEuZW5oYW5jZShhdWRpbywgZGVub2lzZT1GYWxzZSwgYmF0Y2g9RmFsc2UpLmNwdSgpLm51bXB5KCkuc3F1ZWV6ZSgpDQogICAgc2Yud3JpdGUoc3RyKGRzdCksIG91dCwgNDgwMDApDQogICAgcmV0dXJuIFRydWUNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBTdGFnZSAyOiBSTk5vaXNlIGFydGlmYWN0IHJlbW92YWwNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQpkZWYgX3N0YWdlX3Jubm9pc2Uoc3JjLCBkc3QpOg0KICAgIHRtcDQ4ID0gc3RyKGRzdCkgKyAiLl80OGsud2F2Ig0KICAgIHRyeToNCiAgICAgICAgc3VicHJvY2Vzcy5ydW4oDQogICAgICAgICAgICBbImZmbXBlZyIsICIteSIsICItaSIsIHN0cihzcmMpLA0KICAgICAgICAgICAgICItYXIiLCAiNDgwMDAiLCAiLWFjIiwgIjEiLCAiLXNhbXBsZV9mbXQiLCAiczE2IiwgdG1wNDhdLA0KICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9VHJ1ZSwNCiAgICAgICAgKQ0KICAgIGV4Y2VwdCBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcjoNCiAgICAgICAgcmV0dXJuIEZhbHNlDQoNCiAgICBkZW5vaXNlZCA9IHN0cihkc3QpICsgIi5fZG4ud2F2Ig0KICAgIHJhbiA9IEZhbHNlDQogICAgIyB0cnkgQ0xJIGZpcnN0DQogICAgdHJ5Og0KICAgICAgICBzdWJwcm9jZXNzLnJ1bihbImRlbm9pc2UiLCB0bXA0OCwgZGVub2lzZWRdLCBjYXB0dXJlX291dHB1dD1UcnVlLCBjaGVjaz1UcnVlKQ0KICAgICAgICByYW4gPSBUcnVlDQogICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOg0KICAgICAgICBwYXNzDQogICAgIyBmYWxsYmFjazogUHl0aG9uIEFQSQ0KICAgIGlmIG5vdCByYW46DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGZyb20gcHlybm5vaXNlIGltcG9ydCBSTk5vaXNlDQogICAgICAgICAgICBkZW5vaXNlciA9IFJOTm9pc2Uoc2FtcGxlX3JhdGU9NDgwMDApDQogICAgICAgICAgICBmb3IgXyBpbiBkZW5vaXNlci5kZW5vaXNlX3dhdih0bXA0OCwgZGVub2lzZWQpOg0KICAgICAgICAgICAgICAgIHBhc3MNCiAgICAgICAgICAgIHJhbiA9IFRydWUNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIHBhc3MNCiAgICAjIGNsZWFudXAgdGVtcCBpbnB1dHMNCiAgICBmb3IgZiBpbiBbdG1wNDhdOg0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhmKToNCiAgICAgICAgICAgIG9zLnJlbW92ZShmKQ0KICAgIGlmIG5vdCByYW4gb3Igbm90IG9zLnBhdGguZXhpc3RzKGRlbm9pc2VkKToNCiAgICAgICAgcmV0dXJuIEZhbHNlDQoNCiAgICAjIHJlc3RvcmUgc2FtcGxlIHJhdGUNCiAgICB0cnk6DQogICAgICAgIHN1YnByb2Nlc3MucnVuKA0KICAgICAgICAgICAgWyJmZm1wZWciLCAiLXkiLCAiLWkiLCBkZW5vaXNlZCwgIi1hciIsICIyNDAwMCIsIHN0cihkc3QpXSwNCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsIGNoZWNrPVRydWUsDQogICAgICAgICkNCiAgICBleGNlcHQgc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3I6DQogICAgICAgIHNodXRpbC5jb3B5MihkZW5vaXNlZCwgc3RyKGRzdCkpDQogICAgaWYgb3MucGF0aC5leGlzdHMoZGVub2lzZWQpOg0KICAgICAgICBvcy5yZW1vdmUoZGVub2lzZWQpDQogICAgcmV0dXJuIFRydWUNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBTdGFnZSAzOiBhdXRvLWVkaXRvciBzaWxlbmNlIC8gYXJ0aWZhY3QgdHJpbW1pbmcNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tDQpkZWYgX3N0YWdlX2F1dG9lZGl0b3Ioc3JjLCBkc3QpOg0KICAgIHRyeToNCiAgICAgICAgc3VicHJvY2Vzcy5ydW4oDQogICAgICAgICAgICBbImF1dG8tZWRpdG9yIiwgc3RyKHNyYyksICItLW91dHB1dCIsIHN0cihkc3QpLA0KICAgICAgICAgICAgICItLXRocmVzaG9sZCIsICIwLjA0IiwgIi0tbWFyZ2luIiwgIjAuMiJdLA0KICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9VHJ1ZSwNCiAgICAgICAgKQ0KICAgICAgICByZXR1cm4gZHN0LmV4aXN0cygpIGFuZCBkc3Quc3RhdCgpLnN0X3NpemUgPiAwDQogICAgZXhjZXB0IChzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvciwgRmlsZU5vdEZvdW5kRXJyb3IpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KIyBTdGFnZSA0OiBGRm1wZWcgbWFzdGVyaW5nIChFUSwgY29tcHJlc3Npb24sIExVRlMpDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQ0KZGVmIF9zdGFnZV9tYXN0ZXIoc3JjLCBkc3QpOg0KICAgICMgcHJvYmUgc291cmNlIHNhbXBsZSByYXRlDQogICAgdHJ5Og0KICAgICAgICBwcm9iZSA9IHN1YnByb2Nlc3MucnVuKA0KICAgICAgICAgICAgWyJmZnByb2JlIiwgIi12IiwgImVycm9yIiwgIi1zZWxlY3Rfc3RyZWFtcyIsICJhOjAiLA0KICAgICAgICAgICAgICItc2hvd19lbnRyaWVzIiwgInN0cmVhbT1zYW1wbGVfcmF0ZSIsICItb2YiLCAiY3N2PXA9MCIsDQogICAgICAgICAgICAgc3RyKHNyYyldLA0KICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCBjaGVjaz1UcnVlLA0KICAgICAgICApDQogICAgICAgIHNyID0gaW50KHByb2JlLnN0ZG91dC5zdHJpcCgpKSBvciAyNDAwMA0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHNyID0gMjQwMDANCg0KICAgIGNoYWluID0gIiwiLmpvaW4oWw0KICAgICAgICAiaGlnaHBhc3M9Zj04MCIsDQogICAgICAgICJlcXVhbGl6ZXI9Zj0zMDAwOnQ9cTp3PTE6Zz0tMiIsDQogICAgICAgICJlcXVhbGl6ZXI9Zj0xNTA6dD1xOnc9MTpnPTEiLA0KICAgICAgICAiY29tcGFuZD1hdHRhY2tzPTAuMzpkZWNheXM9MC44OnBvaW50cz0tODAvLTgwfC0yMC8tMTR8MC8tNzpnYWluPTAiLA0KICAgICAgICAibG91ZG5vcm09ST0tMTk6VFA9LTEuNTpMUkE9MTEiLA0KICAgIF0pDQogICAgdHJ5Og0KICAgICAgICBzdWJwcm9jZXNzLnJ1bigNCiAgICAgICAgICAgIFsiZmZtcGVnIiwgIi15IiwgIi1pIiwgc3RyKHNyYyksICItYWYiLCBjaGFpbiwNCiAgICAgICAgICAgICAiLWFyIiwgc3RyKHNyKSwgc3RyKGRzdCldLA0KICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9VHJ1ZSwNCiAgICAgICAgKQ0KICAgICAgICByZXR1cm4gVHJ1ZQ0KICAgIGV4Y2VwdCBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcjoNCiAgICAgICAgcmV0dXJuIEZhbHNlDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiMgUHVibGljIEFQSQ0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NClNUQUdFUyA9IFsNCiAgICAoImxhdmEiLCAgICAgICBfc3RhZ2VfbGF2YSksDQogICAgKCJybm5vaXNlIiwgICAgX3N0YWdlX3Jubm9pc2UpLA0KICAgICgiYXV0b2VkaXRvciIsIF9zdGFnZV9hdXRvZWRpdG9yKSwNCiAgICAoIm1hc3RlciIsICAgICBfc3RhZ2VfbWFzdGVyKSwNCl0NCg0KDQpkZWYgZW5oYW5jZV9hdWRpbyhpbnB1dF9wYXRoLCBvdXRwdXRfcGF0aD1Ob25lLCBlbmFibGU9VHJ1ZSk6DQogICAgIiIiUnVuIHRoZSBmdWxsIHBvc3QtcHJvY2Vzc2luZyBwaXBlbGluZS4NCg0KICAgIEFyZ3M6DQogICAgICAgIGlucHV0X3BhdGg6ICBQYXRoIHRvIGlucHV0IFdBVi4NCiAgICAgICAgb3V0cHV0X3BhdGg6IFBhdGggdG8gb3V0cHV0IFdBViAoZGVmYXVsdDogb3ZlcndyaXRlIGlucHV0KS4NCiAgICAgICAgZW5hYmxlOiAgICAgIEZhbHNlIHRvIHNraXAgYWxsIHByb2Nlc3NpbmcuDQoNCiAgICBSZXR1cm5zOg0KICAgICAgICAob3V0cHV0X3BhdGhfc3RyLCBzdGF0c19kaWN0KQ0KICAgICIiIg0KICAgIGlmIG5vdCBlbmFibGU6DQogICAgICAgIHJldHVybiBzdHIoaW5wdXRfcGF0aCksIHt9DQoNCiAgICBpbnB1dF9wYXRoID0gUGF0aChpbnB1dF9wYXRoKQ0KICAgIGlmIG91dHB1dF9wYXRoIGlzIE5vbmU6DQogICAgICAgIG91dHB1dF9wYXRoID0gaW5wdXRfcGF0aA0KICAgIGVsc2U6DQogICAgICAgIG91dHB1dF9wYXRoID0gUGF0aChvdXRwdXRfcGF0aCkNCg0KICAgIHN0YXRzID0ge30NCiAgICBjdXJyZW50ID0gaW5wdXRfcGF0aA0KDQogICAgZm9yIG5hbWUsIGZuIGluIFNUQUdFUzoNCiAgICAgICAgdG1wID0gaW5wdXRfcGF0aC5wYXJlbnQgLyBmIi50bXBfe25hbWV9X3tpbnB1dF9wYXRoLm5hbWV9Ig0KICAgICAgICB0cnk6DQogICAgICAgICAgICB0MCA9IHRpbWUudGltZSgpDQogICAgICAgICAgICBvayA9IGZuKGN1cnJlbnQsIHRtcCkNCiAgICAgICAgICAgIGVsYXBzZWQgPSBmInt0aW1lLnRpbWUoKSAtIHQwOi4xZn1zIg0KICAgICAgICAgICAgaWYgb2sgYW5kIHRtcC5leGlzdHMoKSBhbmQgdG1wLnN0YXQoKS5zdF9zaXplID4gMDoNCiAgICAgICAgICAgICAgICBjdXJyZW50ID0gdG1wDQogICAgICAgICAgICAgICAgc3RhdHNbbmFtZV0gPSBlbGFwc2VkDQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIFtlbmhhbmNlXSB7bmFtZX06IHtlbGFwc2VkfSIpDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIHByaW50KGYiICBbZW5oYW5jZV0ge25hbWV9OiBza2lwcGVkIikNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgcHJpbnQoZiIgIFtlbmhhbmNlXSB7bmFtZX06IGZhaWxlZCAoe2V9KSIpDQoNCiAgICAjIG1vdmUgZmluYWwgcmVzdWx0IHRvIG91dHB1dF9wYXRoDQogICAgaWYgY3VycmVudCAhPSBvdXRwdXRfcGF0aDoNCiAgICAgICAgc2h1dGlsLm1vdmUoc3RyKGN1cnJlbnQpLCBzdHIob3V0cHV0X3BhdGgpKQ0KDQogICAgIyBjbGVhbnVwIHJlbWFpbmluZyB0ZW1wIGZpbGVzDQogICAgZm9yIG5hbWUsIF8gaW4gU1RBR0VTOg0KICAgICAgICB0bXAgPSBpbnB1dF9wYXRoLnBhcmVudCAvIGYiLnRtcF97bmFtZX1fe2lucHV0X3BhdGgubmFtZX0iDQogICAgICAgIGlmIHRtcC5leGlzdHMoKSBhbmQgdG1wICE9IG91dHB1dF9wYXRoOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIHRtcC51bmxpbmsoKQ0KICAgICAgICAgICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgICAgICAgICAgcGFzcw0KDQogICAgcmV0dXJuIHN0cihvdXRwdXRfcGF0aCksIHN0YXRzDQo=').decode('utf-8'))
SRC.joinpath('reel_render.py').write_text(
    base64.b64decode('...').decode('utf-8'))
print('story_src installed:', sorted(p.name for p in SRC.glob('*.py')))


In [ ]:
# Cell 3 — Assets: fetch reels_batch.txt + 10 female voices + outro.
import os, json, subprocess, urllib.request, urllib.parse, shutil
from pathlib import Path

# Reels working dir #######################################################
REEL_ROOT = Path("/content/weekendpulse_reels")
V = REEL_ROOT / "voices"
O = REEL_ROOT / "output"
A = REEL_ROOT / "assets"
for d in (V, O, A):
    d.mkdir(parents=True, exist_ok=True)

REPO = "thechemiluminary/WeekendPulse"
RAW = "https://raw.githubusercontent.com"

# 1. Batch of approved reels (produced by the GitHub reel_batch workflow) ###
def fetch_raw(path):
    url = f"{RAW}/{REPO}/main/{path}"
    with urllib.request.urlopen(url, timeout=40) as r:
        return r.read()

try:
    batch_bytes = fetch_raw("reels_batch.txt")
    (REEL_ROOT / "reels_batch.txt").write_bytes(batch_bytes)
    batch = json.loads(batch_bytes.decode("utf-8"))
    print(f"Loaded reels_batch.txt: {len(batch['reels'])} reel(s)")
except Exception as e:
    print("WARN could not fetch a fresh reels_batch.txt:", e)
    batch = {"generated_at_utc": None, "count": 0, "reels": []}

# 2. Download 10 female voice folders + emotion clips ####################
def dl(url, dest):
    try:
        urllib.request.urlretrieve(url, dest)
    except Exception:
        return False
    return Path(dest).exists() and Path(dest).stat().st_size > 1000

# voices-emotion/<voice>/<emotion>.flac  (OwenTyme/voice-zero, public)
VOICE_REPO = "OwenTyme/voice-zero"
VOICE_PATH = "voices-emotion"
FEMALE_BASE = [
    "kristin_hughes", "jodi_krangle", "karen_savage",
    "emily_cripps", "cori_samuel", "mil_nicholson",
    "amy_koenig", "alana_jordan", "emily_anderson", "anna_simon",
]
# only fetch the emotions this project uses; neutral always
EMOTION_CLIPS = ["neutral", "excited", "surprised"]
got_voices = []
for v in FEMALE_BASE:
    vdir = V / v
    vdir.mkdir(parents=True, exist_ok=True)
    have = 0
    for em in EMOTION_CLIPS:
        dest = vdir / f"{em}.flac"
        url = (f"{RAW}/{VOICE_REPO}/main/{VOICE_PATH}/"
               f"{urllib.parse.quote(v)}/{em}.flac")
        if dl(url, dest):
            have += 1
    if have:
        got_voices.append(v)
        print(f"  voice + {v} ({have}/{len(EMOTION_CLIPS)} clips)")
    else:
        print(f"  MISSING voice {v}")
print(f"Downloaded {len(got_voices)} female voice(s) into {V}: {got_voices}")
if not got_voices:
    raise RuntimeError("No voices downloaded - check VOICE_REPO/path above")

# 3. Outro clip (user's fixed 4s outro) ###################################
outro_src = A / "outro.mp4"
outro_url = f"{RAW}/{REPO}/main/outro/{'outro.mp4'}"
if not outro_src.exists() or outro_src.stat().st_size < 1000:
    dl(outro_url, outro_src)
if outro_src.exists() and outro_src.stat().st_size > 1000:
    print("outro.mp4 ready:", outro_src.stat().st_size // 1024, "KiB")
else:
    print("WARN outro.mp4 missing — reels will render without the outro clip")

print("\nAssets ready. Review reels_batch.txt before rendering.")


In [ ]:
# Cell 4 — RENDER all reels in the batch (one MP4 per reel).
import json, time
from pathlib import Path

REEL_ROOT = Path("/content/weekendpulse_reels")
OUT = REEL_ROOT / "output"

# Load batch built in the Assets cell
with open(REEL_ROOT / "reels_batch.txt", encoding="utf-8") as f:
    batch = json.load(f)

entries = batch.get("reels", [])
entries = entries[:6]                       # soft cap REEL_MAX_PER_RUN=6
print(f"Rendering {len(entries)} reel(s) ...\n")

# the renderer module (written to disk by the 'write files' cell)
import sys
sys.path.insert(0, str(REEL_ROOT / "story_src"))
import reel_render

t_start = time.time()
results = reel_render.render_all(entries)
print("\n=== SUMMARY ===")
for r in results:
    print(f"  {r['slug']:<30} {r['duration_s']:>5.1f}s  voice={r.get('voice','?'):<16} emotion={r['emotion']:<9} music={r['music']}")

# persist summaries for the preview cell
(REEL_ROOT / "rendered_results.json").write_text(
    json.dumps(results, indent=2), encoding="utf-8")
(REEL_ROOT / "rendered_list.txt").write_text(
    "\n".join(str(OUT / r["slug"]) + ".mp4" for r in results), encoding="utf-8")
print(f"\nTotal render time: {time.time() - t_start:.0f}s")
print("Outputs:", OUT)


In [ ]:
# Cell 5 — Preview each rendered reel, switching with a slider (Next/Previous).
# Uses ipywidgets + embedded IPython.display.Video so it plays reliably in Colab.
import json
from pathlib import Path
from IPython.display import display, Video

REEL_ROOT = Path("/content/weekendpulse_reels")
OUT = REEL_ROOT / "output"

# --- order rendered mp4s by slug (render summary order if present)
results = []
res_path = REEL_ROOT / "rendered_results.json"
if res_path.exists():
    with open(res_path, encoding="utf-8") as f:
        results = json.load(f)
paths = [p for r in results
         if (p := OUT / f"{r['slug']}.mp4").exists()]
if not paths:
    paths = sorted(OUT.glob("*.mp4"))

if not paths:
    print("No rendered MP4s yet — run the RENDER cell first.")
else:
    def show(i):
        i = max(0, min(len(paths) - 1, i))
        path = paths[i]
        meta = results[i] if i < len(results) else {}
        print(f"{i+1}/{len(paths)}  {path.name}  "
              f"emotion={meta.get('emotion','?')}  music={meta.get('music','-')}  "
              f"{meta.get('duration_s',0):.1f}s")
        display(Video(filename=str(path), embed=True,
                      metadata={"mimetype": "video/mp4"}, width=360))
        state = REEL_ROOT / "preview_state.json"
        state.write_text(json.dumps({"i": i}), encoding="utf-8")

    try:
        import ipywidgets as widgets
        idx = 0
        s = REEL_ROOT / "preview_state.json"
        if s.exists():
            idx = json.loads(s.read_text(encoding="utf-8")).get("i", 0)
        slider = widgets.IntSlider(min=0, max=len(paths) - 1, value=idx,
                                   description="Reel")
        out = widgets.Output()
        # re-draw on slider change
        def _on_change(change):
            out.clear_output(wait=True)
            with out:
                show(change["new"])
        slider.observe(_on_change, names="value")
        display(widgets.VBox([
            widgets.HBox([widgets.Label("Index:"), slider]),
            out,
        ]))
        with out:
            show(slider.value)
    except Exception as _e:
        # fallback: static preview of the first reel
        print("ipywidgets unavailable — showing first reel.")
        show(0)


In [ ]:
# Cell 6 — Download rendered MP4s (zip them all, plus individual links).
import zipfile
from pathlib import Path
from IPython.display import FileLink, HTML, display

OUT = Path("/content/weekendpulse_reels/output")
mp4s = sorted(str(p) for p in OUT.glob("*.mp4"))
if not mp4s:
    print("No rendered MP4s yet — run the RENDER cell first.")
else:
    zip_path = OUT / "reels.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for m in mp4s:
            z.write(m, arcname=Path(m).name)
    print(f"{len(mp4s)} reel(s) in {zip_path.name}  ({zip_path.stat().st_size//1024} KiB)")
    display(FileLink(str(zip_path), result_html_prefix="Download all: "))
    display(HTML("<h4>Download individually</h4>"))
    for m in mp4s:
        display(FileLink(m))


In [ ]:
# Cell 7 — POST rendered reels to the Facebook Page and record results.
# Requires two secrets at RUNTIME (never hardcoded):
#   FB_PAGE_TOKEN  -> posts each reel via the Graph API `videos` endpoint
#   GH_TOKEN       -> writes the updated manifest.json back to the repo
# Both are entered via Colab secrets / getpass. This cell is idempotent: it
# posts only reels that are rendered and NOT already recorded as posted.
import json
import os
import time
import requests
from pathlib import Path
from getpass import getpass

REEL_ROOT = Path("/content/weekendpulse_reels")
OUT = REEL_ROOT / "output"
REPO = "thechemiluminary/WeekendPulse"
RAW = "https://raw.githubusercontent.com"
GRAPH = "https://graph.facebook.com/v26.0"
MANIFEST = "manifest.json"

FB_PAGE_TOKEN = os.environ.get("FB_PAGE_TOKEN", "") or getpass("FB page token (paste): ")
GH_TOKEN = os.environ.get("GH_TOKEN", "") or getpass("GitHub repo token (paste): ")
FB_PAGE_ID = os.environ.get("FB_PAGE_ID", "1256418564223752")

if not FB_PAGE_TOKEN or not GH_TOKEN:
    raise SystemExit("Both FB_PAGE_TOKEN and GH_TOKEN are required.")

# 1. Load rendered summaries + the batch we rendered.
with open(REEL_ROOT / "rendered_results.json", encoding="utf-8") as f:
    results = json.load(f)
with open(REEL_ROOT / "reels_batch.txt", encoding="utf-8") as f:
    batch = json.load(f)
by_slug = {r["slug"]: r for r in results}

# 2. Pull the current manifest from the repo so we update latest state.
def gh_manifest():
    try:
        r = requests.get(f"{RAW}/{REPO}/main/{MANIFEST}", timeout=30)
        r.raise_for_status()
        return r.json(), None
    except Exception as e:
        return None, str(e)

def gh_update(manifest_entries, sha):
    url = f"https://api.github.com/repos/{REPO}/contents/{MANIFEST}"
    payload = {
        "message": "chore: record posted reels [skip ci]",
        "content": __import__("base64").b64encode(
            json.dumps(manifest_entries, indent=2, ensure_ascii=False).encode("utf-8")
        ).decode("ascii"),
    }
    if sha:
        payload["sha"] = sha
    return requests.put(url, json=payload,
                        headers={"Authorization": f"Bearer {GH_TOKEN}",
                                 "Accept": "application/vnd.github+json",
                                 "User-Agent": "wp"}, timeout=60)

# A slug -> entry map (slug is stable across batch + manifest via same slugger).
manifest_entries, m_err = gh_manifest()
if manifest_entries is None:
    print("WARN could not fetch manifest:", m_err, "- using empty base.")
    manifest_entries = []

if not isinstance(manifest_entries, list):
    manifest_entries = manifest_entries.get("entries", []) if isinstance(manifest_entries, dict) else []

sha = None
try:
    r = requests.get(f"https://api.github.com/repos/{REPO}/contents/{MANIFEST}",
                     headers={"Accept": "application/vnd.github+json",
                              "User-Agent": "wp"}, timeout=30)
    if r.status_code == 200:
        sha = r.json().get("sha")
except Exception:
    sha = None

def slug_for(url, title):
    url = (url or "").strip()
    if url:
        base = url.rstrip("/").rsplit("/", 1)[-1]
        base = "".join(c for c in base if c.isalnum() or c in "-_") or "reel"
        return base[:40]
    t = (title or "").strip().lower()
    return (("-".join(w for w in t.replace("-", " ").split() if w)[:5])[:40]) or "reel"

by_manifest = {}
for e in manifest_entries:
    s = e.get("slug") or slug_for(e.get("url"), e.get("title"))
    by_manifest[s] = e

header = {"Authorization": f"Bearer {FB_PAGE_TOKEN}"}
posted = 0
skipped = 0
failed = []
for ent in batch.get("reels", []):
    slug = ent["slug"]
    summary = by_slug.get(slug)
    mp4 = OUT / f"{slug}.mp4"
    if not mp4.exists():
        print(f"  SKIP {slug}: mp4 not found")
        skipped += 1
        continue

    m = by_manifest.get(slug, {"slug": slug})
    if m.get("reel_posted"):
        print(f"  SKIP {slug}: already posted (id={m.get('reel_video_id')})")
        skipped += 1
        continue

    title = ent.get("title", "")
    desc = ent.get("reel_blurb", "") or title
    print(f"  POST {slug} -> videos ...")
    try:
        with open(mp4, "rb") as vf:
            resp = requests.post(
                f"{GRAPH}/{FB_PAGE_ID}/videos",
                files={"source": ("reel.mp4", vf, "video/mp4")},
                data={"title": title, "description": desc},
                headers=header, timeout=600)
        data = resp.json()
        if resp.status_code == 200 and ("id" in data):
            vid = data["id"]
            m.update({
                "rendered": True,
                "reel_posted": True,
                "reel_video_id": vid,
                "reel_voice": summary.get("voice", "") if summary else "",
                "reel_emotion_used": summary.get("emotion", "") if summary else "",
                "reel_music": summary.get("music") or "" if summary else "",
                "reel_rendered_at": __import__("datetime").datetime.now(
                    __import__("datetime").timezone.utc).isoformat(),
            })
            by_manifest[slug] = m
            posted += 1
            print(f"    posted id={vid}")
        else:
            failed.append((slug, data))
            print(f"    FAIL {slug}: {data}")
    except Exception as e:
        failed.append((slug, str(e)))
        print(f"    ERROR {slug}: {e}")

# 3. Write the manifest back to the repo.
out_list = list(by_manifest.values())
put = gh_update(out_list, sha)
if put.status_code in (200, 201):
    print(f"\nManifest updated on repo ({len(out_list)} entries).")
else:
    print(f"\nManifest write-back FAILED ({put.status_code}): {put.text[:300]}")

print(f"\nPosted: {posted}  Skipped(already): {skipped}  Failed: {len(failed)}")
if failed:
    print("Failed:", json.dumps(failed, default=str))
